# 🏋️ Challenge 07 — Recursion, Algorithms & Data Structures
*Covers Modules 15–16.* 🟢 easy · 🟡 medium · 🔴 hard.

In [1]:
def check(name, test):
    try:
        ok = bool(test())
    except Exception as e:
        ok = False
        name += f"   (raised {type(e).__name__}: {e})"
    print("✅ PASS" if ok else "❌ FAIL", "-", name)

### 🟢 Problem 1 — Recursion warm-up
Write these **recursively** (no loops): `factorial(n)`, `sum_digits(n)`, `power(base, exp)` (exp ≥ 0), `reverse(s)`.

In [2]:
def factorial(n):
    return 1 if n <= 1 else n * factorial(n - 1)

def sum_digits(n):
    return n if n < 10 else n % 10 + sum_digits(n // 10)

def power(base, exp):
    return 1 if exp == 0 else base * power(base, exp - 1)

def reverse(s):
    return s if len(s) <= 1 else reverse(s[1:]) + s[0]

In [3]:
check("factorial", lambda: factorial(0) == 1 and factorial(6) == 720)
check("sum_digits", lambda: sum_digits(12345) == 15 and sum_digits(0) == 0)
check("power", lambda: power(2, 10) == 1024 and power(5, 0) == 1)
check("reverse", lambda: reverse("python") == "nohtyp" and reverse("") == "")

✅ PASS - factorial
✅ PASS - sum_digits
✅ PASS - power
✅ PASS - reverse


### 🟢 Problem 2 — Binary search
Write an iterative `binary_search(sorted_seq, target)` returning the index or `-1`. It must be O(log n): the last test searches a sequence of 10 million items, which a linear scan would crawl through.

In [4]:
def binary_search(a, target):
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if a[mid] == target:
            return mid
        if a[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

In [5]:
data = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
check("found", lambda: binary_search(data, 23) == 5)
check("first & last", lambda: binary_search(data, 2) == 0 and binary_search(data, 91) == 9)
check("missing", lambda: binary_search(data, 7) == -1)
check("empty list", lambda: binary_search([], 1) == -1)
check("fast on 10 million items", lambda: binary_search(range(0, 20_000_000, 2), 19_999_998) == 9_999_999)

✅ PASS - found
✅ PASS - first & last
✅ PASS - missing
✅ PASS - empty list
✅ PASS - fast on 10 million items


### 🟡 Problem 3 — Implement sorting algorithms
Write `insertion_sort(a)` and `merge_sort(a)`. Both return a **new** sorted list and leave the input untouched.

In [6]:
def insertion_sort(a):
    a = a[:]
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
    return a

def merge_sort(a):
    if len(a) <= 1:
        return a[:]
    mid = len(a) // 2
    left, right = merge_sort(a[:mid]), merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    return out + left[i:] + right[j:]

In [7]:
import random
random.seed(5)
sample = [random.randint(-50, 50) for _ in range(200)]
check("insertion_sort", lambda: insertion_sort(sample) == sorted(sample))
check("merge_sort", lambda: merge_sort(sample) == sorted(sample))
check("empty and single", lambda: merge_sort([]) == [] and insertion_sort([7]) == [7])
check("input untouched", lambda: (lambda c: (merge_sort(c), insertion_sort(c), c)[2])(list(sample)) == sample)
check("duplicates", lambda: merge_sort([3, 1, 3, 1]) == [1, 1, 3, 3])

✅ PASS - insertion_sort
✅ PASS - merge_sort
✅ PASS - empty and single
✅ PASS - input untouched
✅ PASS - duplicates


### 🟡 Problem 4 — Subsets & permutations (recursion)
`subsets(items)` → all subsets as lists (any order, tests sort them). `permutations(items)` → all orderings as lists.

<details><summary>💡 Hint</summary>Subsets: subsets of the rest, plus each of those with the first item added. Permutations: pick each item as the first, recurse on the remaining.</details>

In [8]:
def subsets(items):
    if not items:
        return [[]]
    rest = subsets(items[1:])
    return rest + [[items[0]] + r for r in rest]

def permutations(items):
    if len(items) <= 1:
        return [list(items)]
    out = []
    for i, x in enumerate(items):
        for p in permutations(items[:i] + items[i + 1:]):
            out.append([x] + p)
    return out

In [9]:
check("subsets of 3 -> 8", lambda: sorted(map(sorted, subsets([1, 2, 3]))) == sorted([[], [1], [2], [3], [1, 2], [1, 3], [2, 3], [1, 2, 3]]))
check("subsets of empty", lambda: subsets([]) == [[]])
check("permutations of 3 -> 6", lambda: sorted(permutations([1, 2, 3])) == [[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]])
check("permutations of 4 -> 24", lambda: len(permutations([1, 2, 3, 4])) == 24)

✅ PASS - subsets of 3 -> 8
✅ PASS - subsets of empty
✅ PASS - permutations of 3 -> 6
✅ PASS - permutations of 4 -> 24


### 🟡 Problem 5 — Dynamic programming: coin change & stairs
`min_coins(coins, amount)` → fewest coins to make `amount`, or `-1` if impossible.
`ways_to_climb(n)` → number of ways to climb `n` stairs taking 1 or 2 steps at a time.

In [10]:
def min_coins(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1
    return dp[amount] if dp[amount] != INF else -1

def ways_to_climb(n):
    a, b = 1, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

In [11]:
check("min_coins 63 with US coins", lambda: min_coins([1, 5, 10, 25], 63) == 6)
check("greedy trap 6 = 3+3", lambda: min_coins([1, 3, 4], 6) == 2)
check("impossible", lambda: min_coins([2], 3) == -1)
check("zero amount", lambda: min_coins([1], 0) == 0)
check("stairs", lambda: [ways_to_climb(n) for n in (1, 2, 3, 4, 5)] == [1, 2, 3, 5, 8])
check("stairs 40 fast", lambda: ways_to_climb(40) == 165580141)

✅ PASS - min_coins 63 with US coins
✅ PASS - greedy trap 6 = 3+3
✅ PASS - impossible
✅ PASS - zero amount
✅ PASS - stairs
✅ PASS - stairs 40 fast


### 🟡 Problem 6 — Stack applications
`balanced(s)` → `True` if all brackets `()[]{}` in `s` are balanced and properly nested (ignore other characters).
`evaluate_rpn(tokens)` → evaluate a Reverse Polish expression given as a list of strings, e.g. `["2","1","+","3","*"]` → 9. Operators `+ - * /` (use true division).

In [12]:
def balanced(s):
    pairs, stack = {")": "(", "]": "[", "}": "{"}, []
    for ch in s:
        if ch in "([{":
            stack.append(ch)
        elif ch in pairs:
            if not stack or stack.pop() != pairs[ch]:
                return False
    return not stack

def evaluate_rpn(tokens):
    stack = []
    for t in tokens:
        if t in {"+", "-", "*", "/"}:
            b, a = stack.pop(), stack.pop()
            stack.append({"+": a + b, "-": a - b, "*": a * b, "/": a / b}[t])
        else:
            stack.append(float(t))
    return stack[0]

In [13]:
check("balanced ok", lambda: balanced("{[()]}") and balanced("a(b)c[d]") and balanced(""))
check("balanced bad", lambda: not balanced("{[(])}") and not balanced("(") and not balanced(")("))
check("rpn 1", lambda: evaluate_rpn(["2", "1", "+", "3", "*"]) == 9)
check("rpn 2", lambda: evaluate_rpn(["4", "13", "5", "/", "+"]) == 6.6)
check("rpn negative numbers", lambda: evaluate_rpn(["3", "-4", "*"]) == -12)

✅ PASS - balanced ok
✅ PASS - balanced bad
✅ PASS - rpn 1
✅ PASS - rpn 2
✅ PASS - rpn negative numbers


### 🟡 Problem 7 — Queue: hot potato & sliding window maximum
`hot_potato(names, k)` — players sit in a circle holding a potato that starts with the **first** player. It is passed `k` times to the next player, then whoever holds it is eliminated; play continues with the next player. Return the **last remaining** player. Use `collections.deque` (`rotate`).
`sliding_max(nums, k)` — return the maximum of every window of size `k` in O(n) using a deque of indices.

In [14]:
from collections import deque

def hot_potato(names, k):
    q = deque(names)
    while len(q) > 1:
        q.rotate(-k)          # pass the potato k times
        q.popleft()           # holder is out
    return q[0]

def sliding_max(nums, k):
    dq, out = deque(), []                 # dq holds indices of decreasing values
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:
            dq.popleft()
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out

In [15]:
check("hot potato", lambda: hot_potato(["Bill", "David", "Susan", "Jane", "Kent", "Brad"], 7) == "Susan")
check("hot potato k=1", lambda: hot_potato(["a", "b", "c"], 1) == "c")
check("sliding max", lambda: sliding_max([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7])
check("k = 1", lambda: sliding_max([4, 2], 1) == [4, 2])

✅ PASS - hot potato
✅ PASS - hot potato k=1
✅ PASS - sliding max
✅ PASS - k = 1


### 🔴 Problem 8 — Linked list
Implement `class LinkedList` with `append(x)`, `prepend(x)`, `remove(x)` (first occurrence; returns bool), `reverse()` (in place), `to_list()`, `__len__`. Don't use Python lists internally — build real nodes.

In [16]:
class Node:
    def __init__(self, data, next=None):
        self.data, self.next = data, next

class LinkedList:
    def __init__(self):
        self.head, self._size = None, 0

    def append(self, x):
        node = Node(x)
        if not self.head:
            self.head = node
        else:
            cur = self.head
            while cur.next:
                cur = cur.next
            cur.next = node
        self._size += 1

    def prepend(self, x):
        self.head = Node(x, self.head)
        self._size += 1

    def remove(self, x):
        prev, cur = None, self.head
        while cur:
            if cur.data == x:
                if prev:
                    prev.next = cur.next
                else:
                    self.head = cur.next
                self._size -= 1
                return True
            prev, cur = cur, cur.next
        return False

    def reverse(self):
        prev, cur = None, self.head
        while cur:
            cur.next, prev, cur = prev, cur, cur.next
        self.head = prev

    def to_list(self):
        out, cur = [], self.head
        while cur:
            out.append(cur.data)
            cur = cur.next
        return out

    def __len__(self):
        return self._size

In [17]:
ll = LinkedList()
for x in (2, 3, 4): ll.append(x)
ll.prepend(1)
check("append/prepend", lambda: ll.to_list() == [1, 2, 3, 4] and len(ll) == 4)
check("remove middle", lambda: ll.remove(3) is True and ll.to_list() == [1, 2, 4])
check("remove head", lambda: ll.remove(1) is True and ll.to_list() == [2, 4])
check("remove missing", lambda: ll.remove(99) is False and len(ll) == 2)
def _rev():
    ll.append(8); ll.reverse(); return ll.to_list()
check("reverse", lambda: _rev() == [8, 4, 2])
check("empty list ops", lambda: (lambda e: (e.reverse(), e.to_list())[1])(LinkedList()) == [])

✅ PASS - append/prepend
✅ PASS - remove middle
✅ PASS - remove head
✅ PASS - remove missing
✅ PASS - reverse
✅ PASS - empty list ops


### 🔴 Problem 9 — Binary Search Tree
Implement `BST` with `insert(key)`, `contains(key)`, `inorder()` (sorted list), `height()` (empty tree → 0, single node → 1) and `min()`/`max()`. Ignore duplicates.

In [18]:
class BST:
    class _Node:
        def __init__(self, key):
            self.key, self.left, self.right = key, None, None

    def __init__(self):
        self.root = None

    def insert(self, key):
        def _ins(node):
            if node is None:
                return BST._Node(key)
            if key < node.key:
                node.left = _ins(node.left)
            elif key > node.key:
                node.right = _ins(node.right)
            return node
        self.root = _ins(self.root)

    def contains(self, key):
        node = self.root
        while node:
            if key == node.key:
                return True
            node = node.left if key < node.key else node.right
        return False

    def inorder(self):
        out = []
        def walk(n):
            if n:
                walk(n.left); out.append(n.key); walk(n.right)
        walk(self.root)
        return out

    def height(self):
        def h(n):
            return 0 if n is None else 1 + max(h(n.left), h(n.right))
        return h(self.root)

    def min(self):
        n = self.root
        while n.left:
            n = n.left
        return n.key

    def max(self):
        n = self.root
        while n.right:
            n = n.right
        return n.key

In [19]:
t = BST()
for k in (50, 30, 70, 20, 40, 60, 80, 30):
    t.insert(k)
check("inorder sorted, duplicates ignored", lambda: t.inorder() == [20, 30, 40, 50, 60, 70, 80])
check("contains", lambda: t.contains(60) and not t.contains(65))
check("height", lambda: t.height() == 3)
check("min / max", lambda: t.min() == 20 and t.max() == 80)
check("empty tree height", lambda: BST().height() == 0)

✅ PASS - inorder sorted, duplicates ignored
✅ PASS - contains
✅ PASS - height
✅ PASS - min / max
✅ PASS - empty tree height


### 🔴 Problem 10 — Graph algorithms
Given an adjacency dict `graph = {"A": ["B","C"], ...}` (undirected, listed both ways):
- `bfs_shortest_path(graph, start, goal)` → list of nodes on a shortest path or `None`
- `connected_components(graph)` → number of connected components
- `has_path(graph, a, b)` → bool

In [20]:
from collections import deque

def bfs_shortest_path(graph, start, goal):
    prev, q = {start: None}, deque([start])
    while q:
        node = q.popleft()
        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = prev[node]
            return path[::-1]
        for nb in graph[node]:
            if nb not in prev:
                prev[nb] = node
                q.append(nb)
    return None

def connected_components(graph):
    seen, count = set(), 0
    for n in graph:
        if n not in seen:
            count += 1
            stack = [n]
            while stack:
                cur = stack.pop()
                if cur in seen:
                    continue
                seen.add(cur)
                stack.extend(graph[cur])
    return count

def has_path(graph, a, b):
    return bfs_shortest_path(graph, a, b) is not None

In [21]:
G = {"A": ["B", "C"], "B": ["A", "D"], "C": ["A", "D"], "D": ["B", "C", "E"], "E": ["D"], "X": ["Y"], "Y": ["X"]}
check("shortest path length", lambda: len(bfs_shortest_path(G, "A", "E")) == 4)
check("path endpoints", lambda: (lambda p: (p[0], p[-1]))(bfs_shortest_path(G, "A", "E")) == ("A", "E"))
check("same node", lambda: bfs_shortest_path(G, "A", "A") == ["A"])
check("unreachable", lambda: bfs_shortest_path(G, "A", "X") is None)
check("components", lambda: connected_components(G) == 2)
check("has_path", lambda: has_path(G, "B", "E") and not has_path(G, "E", "Y"))

✅ PASS - shortest path length
✅ PASS - path endpoints
✅ PASS - same node
✅ PASS - unreachable
✅ PASS - components
✅ PASS - has_path


### 🔴 Problem 11 — Two pointers, sliding window, Kadane
- `pair_with_sum(sorted_nums, target)` → tuple of two values summing to target (two pointers), or `None`
- `longest_unique_substring(s)` → length of the longest substring without repeating characters (sliding window)
- `max_subarray(nums)` → largest sum of a contiguous non-empty subarray (Kadane)

In [22]:
def pair_with_sum(nums, target):
    i, j = 0, len(nums) - 1
    while i < j:
        s = nums[i] + nums[j]
        if s == target:
            return nums[i], nums[j]
        if s < target:
            i += 1
        else:
            j -= 1
    return None

def longest_unique_substring(s):
    last, start, best = {}, 0, 0
    for i, ch in enumerate(s):
        if ch in last and last[ch] >= start:
            start = last[ch] + 1
        last[ch] = i
        best = max(best, i - start + 1)
    return best

def max_subarray(nums):
    best = cur = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best

In [23]:
check("pair_with_sum", lambda: pair_with_sum([1, 3, 4, 6, 8, 11], 14) == (3, 11) and pair_with_sum([1, 2], 10) is None)
check("longest unique", lambda: [longest_unique_substring(x) for x in ("abcabcbb", "bbbbb", "pwwkew", "", "abba")] == [3, 1, 3, 0, 2])
check("kadane", lambda: max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6)
check("kadane all negative", lambda: max_subarray([-3, -1, -2]) == -1)

✅ PASS - pair_with_sum
✅ PASS - longest unique
✅ PASS - kadane
✅ PASS - kadane all negative


### 🔴 Problem 12 — LRU cache
Implement `LRUCache(capacity)` with `get(key)` (returns value or `-1`; marks as recently used) and `put(key, value)` (evicts the **least recently used** item when over capacity). Use `collections.OrderedDict`.

In [24]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.cap, self.data = capacity, OrderedDict()

    def get(self, key):
        if key not in self.data:
            return -1
        self.data.move_to_end(key)
        return self.data[key]

    def put(self, key, value):
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.cap:
            self.data.popitem(last=False)

In [25]:
c = LRUCache(2)
c.put(1, "one"); c.put(2, "two")
check("get existing", lambda: c.get(1) == "one")
c.put(3, "three")            # evicts key 2 (1 was just used)
check("evicts least recently used", lambda: c.get(2) == -1)
check("others survive", lambda: c.get(1) == "one" and c.get(3) == "three")
c.put(1, "uno")
check("update existing", lambda: c.get(1) == "uno")
c.put(4, "four")             # evicts 3
check("second eviction", lambda: c.get(3) == -1 and c.get(4) == "four")

✅ PASS - get existing
✅ PASS - evicts least recently used
✅ PASS - others survive
✅ PASS - update existing
✅ PASS - second eviction


---
## 🏁 Next
≥ 90 %? Go to `08_NumPy_Pandas_challenges.ipynb`. **Bonus:** implement Dijkstra with `heapq`, a trie for autocomplete, and quicksort with random pivot; measure them with `timeit`.